In [189]:
import pandas as pd
import numpy as np
import sys, os
from importlib import reload
import yaml
from pathlib import Path
from dataclasses import asdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from analysis import plots
import analysis.report
import training.gbm_model_trainer
from training.gbm_model_trainer import GBMModelTrainerConfig 

reload(analysis)
reload(analysis.plots)
reload(analysis.report)
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\training\\gbm_model_trainer.py'>

## Load data

In [190]:
df = pd.read_csv("../data/bank-full.csv", delimiter=';')

In [191]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [192]:
# Convert 'y' to numeric for analysis
df["target"] = np.where(df["y"] == "no", 0, 1)

# Add weights to test weight parameter across functions 
df["weights"] = np.abs(np.random.randn(len(df.index)))

# Add an arbitrary data split for testing functions (TVH = 40/30/30)
df["random"] = np.random.uniform(0, 1, len(df.index))
df["split"] = np.where(
    df["random"]  > 0.70, 
    "V",
    np.where(
        df["random"]  > 0.40, 
        "H", 
        "T"
    )
)

In [193]:
df["split"].value_counts(dropna=False, normalize=True)

split
T    0.399704
H    0.301121
V    0.299175
Name: proportion, dtype: float64

In [194]:
df.groupby("split")["target"].mean()

split
H    0.117232
T    0.111283
V    0.124353
Name: target, dtype: float64

In [195]:
# Split data; define features and target
train = df.query("split == 'T'").drop(columns=["y", "split"])
test = df.query("split == 'V'").drop(columns=["y", "split"])
holdout = df.query("split == 'H'").drop(columns=["y", "split"])

## Use ModelTrainer class for training
- Can use example config files in `analysis-tools/examples` or create a config within your notebook and save it locally

In [ ]:
# Define your config
from training.gbm_model_trainer import TrainingConfig, EvaluationConfig, TuningConfig  

config = TrainingConfig(
    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",
    output_dir="outputs",
    log_file="training.log",
    model_file="model_obj.json",

    hyperparameters={
        "objective": "binary:logistic",
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
    },

    feval="logloss",
    output_log=True,
    base_margin_col=None,

    # Logging and early stopping
    log_eval_period=50,
    early_stopping_rounds=20,
    early_stopping_metric="logloss",
    early_stopping_maximize=False,

    # Evaluation and reporting
    evaluation=EvaluationConfig(
        output_report=True,
        report_file="model_analysis.html",
        report_params={},
        tabulate_vars=["job", "education", "age"],
        plots_to_add=[
            {"plot": "gain_curve_with_gini", "title": "Gain Curve / Lorenz Curve"},
            {
                "plot": "partial_gini_plot",
                "title": "Partial Gini (Top 15%)",
                "kwargs": {"top_percent": 15},
            },
            {"plot": "lift_chart", "title": "Lift Chart"},
            {"plot": "crunched_residual_plot", "title": "Crunched Residuals"},
            {
                "plot": "plot_residual_fit",
                "title": "Std and Avg of Normalized Residuals",
                "kwargs": {"residual_type": "normalized"},
            },
        ]
    ),

    # Tuning setup
    tuning=TuningConfig(
        search_space={
            "max_depth": [3, 6],
            "eta": {"type": "real", "low": 0.01, "high": 0.3},
            "subsample": [0.6, 0.9],
        },
        n_iter=25,
        output_tuning=True,
        tuning_file="tuning_results.csv",
    )
)

# Output config to YAML
config_output_path = Path("../examples/example_config.yaml")

import yaml
# Write config to YAML file
with config_output_path.open("w") as f:
    yaml.dump(asdict(config), f, indent=2)

print(f"Config saved to {config_output_path.resolve()}")

Config saved to C:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\examples\example_config.yaml


In [298]:
reload(training.gbm_model_trainer)
reload(analysis.report)
reload(analysis.plots)
from scoring import callbacks, scorers
reload(callbacks)
reload(scorers)

<module 'scoring.scorers' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\scoring\\scorers.py'>

In [ ]:
from training.gbm_model_trainer import XGBModelTrainer

mt_xgboost = XGBModelTrainer(
    config_path="../examples/example_config.yaml",
    train_df=train,
    valid_df=test,
    holdout_df=holdout
)

In [275]:
config.evaluation

EvaluationConfig(output_report=True, report_file='model_analysis.html', report_params={}, plots_to_add=[{'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals', 'kwargs': {'residual_type': 'normalized'}}], tabulate_vars=['job', 'education', 'age'])

In [301]:
mt_xgboost.train()

In [302]:
mt_xgboost.evaluate()

c:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\src\analysis\plots.py:636: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
c:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\src\analysis\plots.py:636: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
c:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\src\analysis\plots.py:636: 

Analysis report generated at outputs\model_analysis.html


In [303]:
mt_xgboost.tune()

,eta,max_depth,subsample,score
0,0.010439,6,0.832172,-0.860385
1,0.010000,5,0.817273,-0.859705
2,0.010000,5,0.824249,-0.859506
3,0.026925,6,0.899865,-0.858281
4,0.246533,5,0.780017,-0.858210
5,0.276225,5,0.899919,-0.857416
6,0.024278,6,0.894775,-0.857127
7,0.142513,5,0.885358,-0.856833
8,0.018127,6,0.766198,-0.856286
9,0.268688,4,0.787404,-0.856020


In [304]:
mt_xgboost.log_lines

['2025-07-31 15:07:27,752 [INFO] ---------------------------------------------------------------------------',
 '2025-07-31 15:07:27,752 [INFO] BEGINNING MODEL EVALUATION',
 '2025-07-31 15:07:31,960 [INFO] ---------------------------------------------------------------------------',
 '2025-07-31 15:07:31,960 [INFO] BEGINNING MODEL TRAINING',
 '2025-07-31 15:07:31,994 [INFO] [0] validation-logloss: 0.34891',
 '2025-07-31 15:07:32,064 [INFO] [50] validation-logloss: 0.23257',
 '2025-07-31 15:07:32,115 [INFO] Training completed in 0.1546 seconds',
 '2025-07-31 15:07:32,116 [INFO] Model: XGBoost Booster',
 "2025-07-31 15:07:32,116 [INFO] Hyperparameters: {'colsample_bytree': 0.5, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 42, 'subsample': 0.5}",
 '2025-07-31 15:07:32,116 [INFO] Training data shape: (18071, 19)',
 '2025-07-31 15:07:32,116 [INFO] Validation data shape: (13526, 19)',
 '2025-07-31 15:07:32,116 [INFO] Number of pre

## Test Hyperparameter tuning